# Body Structure Analyzer using MediaPipe Pose Landmarker

Detects body landmarks from a local image file and classifies the body shape
(e.g. Athletic / Broad Shoulder, Pear, Hourglass, Rectangle) based on
shoulder width, hip width, and height ratios.

In [ ]:
# Install required libraries
!pip install -q mediapipe opencv-python matplotlib

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

## Download Pose Landmarker Model

This pre-trained model already knows human body structure and joint locations — no training needed.

In [ ]:
import urllib.request, os

model_path = "pose_landmarker.task"
if not os.path.exists(model_path):
    url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task"
    urllib.request.urlretrieve(url, model_path)
    print("Model downloaded.")
else:
    print("Model already exists, skipping download.")

## Load Image

Set `IMAGE_PATH` to the path of your full-body image file.

In [ ]:
IMAGE_PATH = "your_image.jpg"  # <-- change this to your image file path

image = cv2.imread(IMAGE_PATH)

if image is None:
    raise ValueError(f"Could not read image from path: {IMAGE_PATH}\nMake sure the file exists and the path is correct.")

print(f"Image loaded: {image.shape[1]}x{image.shape[0]} pixels")

## Detect Pose Landmarks

In [ ]:
# Convert BGR (OpenCV default) -> RGB (MediaPipe requirement)
rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
h, w, _ = image.shape

# Wrap in a MediaPipe Image
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

# Configure and create the pose landmarker
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=False
)

detector = vision.PoseLandmarker.create_from_options(options)

# Run inference
result = detector.detect(mp_image)

if not result.pose_landmarks:
    raise RuntimeError("No human body detected in the image. Please use a clear full-body photo.")

print(f"Detected {len(result.pose_landmarks)} person(s)")

## Extract Key Landmark Coordinates

In [ ]:
# MediaPipe landmark indices (standard 33-point skeleton)
NOSE           = 0
LEFT_SHOULDER  = 11
RIGHT_SHOULDER = 12
LEFT_HIP       = 23
RIGHT_HIP      = 24
LEFT_ANKLE     = 27
RIGHT_ANKLE    = 28

landmarks = result.pose_landmarks[0]  # first person

def get_point(index):
    """Convert normalised landmark (0-1) to pixel coordinates."""
    lm = landmarks[index]
    return np.array([int(lm.x * w), int(lm.y * h)])

left_shoulder  = get_point(LEFT_SHOULDER)
right_shoulder = get_point(RIGHT_SHOULDER)
left_hip       = get_point(LEFT_HIP)
right_hip      = get_point(RIGHT_HIP)
left_ankle     = get_point(LEFT_ANKLE)
right_ankle    = get_point(RIGHT_ANKLE)
nose           = get_point(NOSE)

## Calculate Body Measurements

In [ ]:
# Euclidean distances
shoulder_width = np.linalg.norm(left_shoulder - right_shoulder)
hip_width      = np.linalg.norm(left_hip - right_hip)

# Mid-ankle point for height measurement
ankle_mid   = (left_ankle + right_ankle) / 2
body_height = np.linalg.norm(nose - ankle_mid)

# Guard against division by zero
shoulder_hip_ratio = shoulder_width / hip_width if hip_width > 0 else float('inf')

# Body shape classification
if shoulder_hip_ratio > 1.3:
    shape = "Athletic / Broad Shoulder"
elif shoulder_hip_ratio < 0.85:
    shape = "Pear Shape"
elif 0.9 <= shoulder_hip_ratio <= 1.1:
    shape = "Hourglass / Rectangle"
else:
    shape = "Average Build"

## Draw Landmarks & Display Result

In [ ]:
# Work on a copy so the original is not mutated between runs
output_image = image.copy()

# Shoulder line (blue)
cv2.line(output_image, tuple(left_shoulder), tuple(right_shoulder), (255, 0, 0), 3)
# Hip line (red)
cv2.line(output_image, tuple(left_hip), tuple(right_hip), (0, 0, 255), 3)
# Height line (yellow)
cv2.line(output_image, tuple(nose), tuple(ankle_mid.astype(int)), (0, 255, 255), 3)

# Landmark dots
for pt in [left_shoulder, right_shoulder, left_hip, right_hip, nose]:
    cv2.circle(output_image, tuple(pt), 6, (0, 255, 0), -1)

# Shape label
cv2.putText(
    output_image, shape, (20, 50),
    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2
)

# Display inline in VS Code using matplotlib (replaces cv2_imshow)
plt.figure(figsize=(8, 10))
plt.imshow(cv2.cvtColor(output_image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"Body Shape: {shape}")
plt.tight_layout()
plt.show()

## Body Analysis Report

In [ ]:
print("\n========== BODY ANALYSIS ==========")
print(f"Shoulder Width     : {shoulder_width:.2f} px")
print(f"Hip Width          : {hip_width:.2f} px")
print(f"Body Height        : {body_height:.2f} px")
print(f"Shoulder/Hip Ratio : {shoulder_hip_ratio:.2f}")
print(f"Predicted Shape    : {shape}")